In [1]:
import ee
import geemap
import xarray as xr

In [2]:
ee.Authenticate()
ee.Initialize(
    project='ee-gabriel-495521',
    opt_url='https://earthengine-highvolume.googleapis.com'
)

In [3]:
map = geemap.Map()
map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

In [5]:
roi = map.draw_last_feature.geometry()
roi

ee.Geometry({
  "functionInvocationValue": {
    "functionName": "Feature.geometry",
    "arguments": {
      "feature": {
        "functionInvocationValue": {
          "functionName": "Feature",
          "arguments": {
            "geometry": {
              "functionInvocationValue": {
                "functionName": "GeometryConstructors.Polygon",
                "arguments": {
                  "coordinates": {
                    "constantValue": [
                      [
                        [
                          43.417969,
                          36.031332
                        ],
                        [
                          43.417969,
                          42.261049
                        ],
                        [
                          50.888672,
                          42.261049
                        ],
                        [
                          50.888672,
                          36.031332
                        ],
                        [
                          43.417969,
                          36.031332
                        ]
                      ]
                    ]
                  },
                  "geodesic": {
                    "constantValue": false
                  }
                }
              }
            }
          }
        }
      }
    }
  }
})

In [6]:
temp = (
    ee.ImageCollection("NASA/VIIRS/002/VNP21A1D")
    .filterDate('2020', '2021')
    .select('LST_1KM')
)
temp

In [7]:
ndvi = (
    ee.ImageCollection("NASA/VIIRS/002/VNP13A1")
    .filterDate('2020', '2021')
    .select('NDVI')
)

ndvi

In [8]:
ds_temp = xr.open_dataset(
    temp,
    engine='ee',
    crs = 'EPSG:4326',
    geometry = roi,
    scale = 0.01
)

ds_temp

<xarray.Dataset> Size: 681MB
Dimensions:  (time: 366, lon: 747, lat: 623)
Coordinates:
  * time     (time) datetime64[ms] 3kB 2020-01-01 2020-01-02 ... 2020-12-31
  * lon      (lon) float64 6kB 43.42 43.43 43.44 43.45 ... 50.86 50.87 50.88
  * lat      (lat) float64 5kB 36.04 36.05 36.06 36.07 ... 42.24 42.25 42.26
Data variables:
    LST_1KM  (time, lon, lat) float32 681MB ...
Attributes:
    crs:      EPSG:4326

In [10]:
ds_ndvi = xr.open_dataset(
    ndvi,
    engine='ee',
    crs='EPSG:4326',
    geometry=roi,
    scale=0.005
)

ndvi

In [11]:
ds_temp=ds_temp.sortby('time') * 1
ds_ndvi=ds_ndvi.sortby('time') * 1

In [13]:
# Converter para o escala mensal
ds_temp_monthly = ds_temp.resample(time='ME').mean('time')
ds_ndvi_monthly = ds_ndvi.resample(time='ME').median('time')

In [14]:
ds_temp_monthly

<xarray.Dataset> Size: 22MB
Dimensions:  (time: 12, lon: 747, lat: 623)
Coordinates:
  * time     (time) datetime64[ms] 96B 2020-01-31 2020-02-29 ... 2020-12-31
  * lon      (lon) float64 6kB 43.42 43.43 43.44 43.45 ... 50.86 50.87 50.88
  * lat      (lat) float64 5kB 36.04 36.05 36.06 36.07 ... 42.24 42.25 42.26
Data variables:
    LST_1KM  (time, lon, lat) float32 22MB 289.4 289.2 288.9 ... 282.9 282.9
Attributes:
    crs:      EPSG:4326

In [15]:
ds_ndvi_monthly

<xarray.Dataset> Size: 89MB
Dimensions:  (time: 12, lon: 1494, lat: 1246)
Coordinates:
  * time     (time) datetime64[ms] 96B 2020-01-31 2020-02-29 ... 2020-12-31
  * lon      (lon) float64 12kB 43.42 43.43 43.43 43.44 ... 50.88 50.88 50.89
  * lat      (lat) float64 10kB 36.03 36.04 36.04 36.05 ... 42.25 42.25 42.26
Data variables:
    NDVI     (time, lon, lat) float32 89MB 0.346 0.3119 0.2686 ... nan nan nan
Attributes:
    crs:      EPSG:4326

In [16]:
# Match datasets espacialmente
ds_ndvi_monthly_regrid = ds_ndvi_monthly.interp_like(ds_temp_monthly)
ds_ndvi_monthly_regrid

<xarray.Dataset> Size: 45MB
Dimensions:  (time: 12, lon: 747, lat: 623)
Coordinates:
  * time     (time) datetime64[ms] 96B 2020-01-31 2020-02-29 ... 2020-12-31
  * lon      (lon) float64 6kB 43.42 43.43 43.44 43.45 ... 50.86 50.87 50.88
  * lat      (lat) float64 5kB 36.04 36.05 36.06 36.07 ... 42.24 42.25 42.26
Data variables:
    NDVI     (time, lon, lat) float64 45MB 0.3143 0.2619 0.2947 ... nan nan nan
Attributes:
    crs:      EPSG:4326

In [17]:
ds_temp_monthly


<xarray.Dataset> Size: 22MB
Dimensions:  (time: 12, lon: 747, lat: 623)
Coordinates:
  * time     (time) datetime64[ms] 96B 2020-01-31 2020-02-29 ... 2020-12-31
  * lon      (lon) float64 6kB 43.42 43.43 43.44 43.45 ... 50.86 50.87 50.88
  * lat      (lat) float64 5kB 36.04 36.05 36.06 36.07 ... 42.24 42.25 42.26
Data variables:
    LST_1KM  (time, lon, lat) float32 22MB 289.4 289.2 288.9 ... 282.9 282.9
Attributes:
    crs:      EPSG:4326

In [ ]:
# Empacotar os dois datasets
ds_merge = xr.merge(ds_temp_monthly, ds_ndvi_monthly_regrid)
ds_merge